# Pyneath QLoRA — Colab Free
Candidate environment; GPU verification has not been performed. Use only reviewed fictional data. This notebook trains offline; it does not host the product or publish artifacts. Run cells manually, in order. No paid fallback.

Prepare a private checkout at an approved commit before continuing. Do not paste credentials into notebook cells. Colab processes uploaded data on Google infrastructure.

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT = Path("/content/project.beneath-the-pine")
COMMIT = "REPLACE_WITH_APPROVED_40_CHARACTER_COMMIT"
assert len(COMMIT) == 40 and PROJECT.is_dir()
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip() == COMMIT
%cd /content/project.beneath-the-pine

## Install the pinned candidate environment
Restart the runtime after installation if requested, then rerun the checkout/path cell. Dependency resolution and `pip check` are necessary but do not establish GPU compatibility.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "ml/training/requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

## Build the shared contracts
Node >=22.12 and pnpm 10.32.1 must be installed in this runtime. No dataset content is sent to npm. These commands build the same schemas used by NestJS.

In [ ]:
subprocess.run(["pnpm", "install", "--frozen-lockfile"], check=True)
subprocess.run(["pnpm", "--filter", "@pyneath/contracts", "build"], check=True)
subprocess.run(["node", "ml/scripts/export_protocol.cjs", "--check"], check=True)

## Explicit private data/checkpoint locations
Mount your own Drive only after authorizing that action. Place reviewed train/validation files and their frozen manifest in the directory below. Keep the final holdout separate. Raw data and artifacts must not be committed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PRIVATE = Path("/content/drive/MyDrive/pyneath-private")
DATA = PRIVATE / "reviewed-v2"
STORE = PRIVATE / "checkpoints"
CONFIG = PROJECT / "ml/configs/qlora-v2.yaml"
common = ["--config", str(CONFIG), "--train", str(DATA / "train.jsonl"), "--validation", str(DATA / "validation.jsonl"), "--manifest", str(DATA / "manifest.json"), "--checkpoint-store", str(STORE)]

## Mandatory smoke run
Stops after 20 optimizer steps. Requires reviewed data. The command fails before loading a model if dependencies, contract or dataset identity do not match.

In [ ]:
SMOKE = Path("/content/pyneath-smoke-v2")
subprocess.run([sys.executable, "-m", "ml.training.train", *common, "--smoke", "--output", str(SMOKE)], check=True)
import json
assert json.loads((SMOKE / "result.json").read_text())["status"] == "smoke-passed"

## First full candidate
At most two epochs and two hours per configuration, including resume time. Select candidates using validation, not the final test set. Re-running into the same output directory is refused.

In [ ]:
RUN = Path("/content/pyneath-candidate-v2-lr1e4")
subprocess.run([sys.executable, "-m", "ml.training.train", *common, "--output", str(RUN), "--smoke-evidence", str(SMOKE / "result.json")], check=True)

## Resume after disconnection
Use a durable checkpoint containing `complete.json`. Restore the matching smoke result and configuration first. Incomplete checkpoints, changed hashes or exhausted time budgets are rejected. Uncomment only when resuming.

In [ ]:
# CHECKPOINT = STORE / RUN.name / "checkpoint-100"
# subprocess.run([sys.executable, "-m", "ml.training.train", *common, "--output", str(RUN), "--smoke-evidence", str(SMOKE / "result.json"), "--resume", str(CHECKPOINT)], check=True)

## Evaluation and handoff
Follow `docs/05-machine-learning/local-ai-runbook.md`: collect baseline/adapter validation predictions, obtain blind human scoring, freeze candidate selection, then open the independent final holdout. Export only a passed adapter. A successful train is not approval to deploy. Copy result/adapter/environment lock to private storage before the runtime expires.